# Transformer Summarization - Evaluation

Evaluation of fine-tuned summarization model using ROUGE and BERTScore metrics.

## Part-2:
### Installing and importing libraries

In [16]:
!pip install -q transformers datasets evaluate rouge_score bert_score torch pandas accelerate sentencepiece

import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer)
from datasets import Dataset, DatasetDict
from evaluate import load
from tqdm import tqdm
import warnings, random
warnings.filterwarnings('ignore')

# checking GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## Loading and splitting the dataset

In [17]:
try: # loading the essays dataset
    df = pd.read_csv('/content/essays.csv')
    print("Successfully loaded")
except FileNotFoundError:
    print("File not found")
    raise

print(f"\nTotal records: {df.shape[0]}")
print(f"Columns: {df.columns.tolist()}")
assert 'essay' in df.columns and 'description' in df.columns # confirming required columns
train_df = df.iloc[:1600].copy() # splitting data
val_df   = df.iloc[1600:1800].copy()
test_df  = df.iloc[1800:].copy()
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
raw_datasets = DatasetDict({ # converting to HuggingFace datasetdict
    "train": Dataset.from_pandas(train_df[['essay', 'description']]),
    "validation": Dataset.from_pandas(val_df[['essay', 'description']]),
    "test": Dataset.from_pandas(test_df[['essay', 'description']])})

Successfully loaded

Total records: 2235
Columns: ['title', 'description', 'essay', 'authors', 'source_url', 'thumbnail_url']
Train: 1600 | Validation: 200 | Test: 435


## Preprocessing and tokenization

In [18]:
MODEL_NAME = "google-t5/t5-small" # setting up the model and tokenization parameters
PREFIX = "summarize: "
MAX_INPUT_LEN = 1024 # maximum length for input essays
MAX_TARGET_LEN = 128 # maximum length for output summaries

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess_function(examples): # Tokenizing essay and description with the prefix 'summarize:'
    inputs = [PREFIX + text for text in examples["essay"]]
    model_inputs = tokenizer( # tokenizing the input essays
        inputs, max_length=MAX_INPUT_LEN,
        truncation=True, padding="max_length")
    labels = tokenizer( # tokenizing the target summaries
        examples["description"], max_length=MAX_TARGET_LEN,
        truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets...")
tokenized_datasets = raw_datasets.map( # applying preprocessing to all datasets
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names, # removing original columns
    desc="Running tokenizer")

Loading tokenizer for google-t5/t5-small...
Tokenizing datasets...


Running tokenizer:   0%|          | 0/1600 [00:00<?, ? examples/s]

Running tokenizer:   0%|          | 0/200 [00:00<?, ? examples/s]

Running tokenizer:   0%|          | 0/435 [00:00<?, ? examples/s]

## Model, collator, and metric function

In [19]:
print(f"Loading base model: {MODEL_NAME}") # preparing model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True) # preparing data collator
rouge_metric = load("rouge")

def compute_metrics(eval_pred): # defining metric function,computes ROUGE during training
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {"rouge1": rouge["rouge1"], "rouge2": rouge["rouge2"], "rougeL": rouge["rougeL"]}


Loading base model: google-t5/t5-small


## Training setup

In [21]:
print("Configuring training parameters...")

training_args = Seq2SeqTrainingArguments( # setting up all the training parameters
    output_dir="./results",
    num_train_epochs=3, # training for 3 epochs
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=50,         # log every 50 steps
    logging_strategy="steps")

trainer = Seq2SeqTrainer( # setting up the trainer
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

print("Starting fine‑tuning for 3 epochs...")
train_result = trainer.train() # starting the training process
print("Fine‑tuning completed.")
finetuned_path = "./t5-small-aeon-finetuned"
trainer.save_model(finetuned_path) # saving the fine tuned model
tokenizer.save_pretrained(finetuned_path)
print(f"Fine‑tuned model saved at {finetuned_path}")

Configuring training parameters...
Starting fine‑tuning for 3 epochs...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,0.844400,0.783637,0.058998,0.006797,0.045867
2,0.791800,0.771402,0.168555,0.019553,0.132750
3,0.788100,0.768957,0.168606,0.018449,0.133543


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Fine‑tuning completed.
Fine‑tuned model saved at ./t5-small-aeon-finetuned


## Evaluation on test set

In [22]:
print("Loading fine‑tuned and original models...")
ft_model = AutoModelForSeq2SeqLM.from_pretrained(finetuned_path).to(device)# loading my fine tuned model
ft_tokenizer = AutoTokenizer.from_pretrained(finetuned_path)
# loading the T5-small model for comparison
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

test_texts = raw_datasets["test"]["essay"] # getting test data
references = raw_datasets["test"]["description"]

def generate_summaries(model, tokenizer, texts, batch_size=8): # generates summaries in small batches
    summaries = []
    model.eval()
    for i in tqdm(range(0, len(texts), batch_size)): # processing in batches with progress bar
        batch = [PREFIX + t for t in texts[i:i+batch_size]]  # getting current batch
        encoded = tokenizer(batch, max_length=MAX_INPUT_LEN, truncation=True,
                            padding=True, return_tensors="pt").to(device) # tokenizing the batch
        with torch.no_grad(): # generating summaries
            out = model.generate(**encoded, max_length=MAX_TARGET_LEN, num_beams=4, early_stopping=True)
        summaries.extend(tokenizer.batch_decode(out, skip_special_tokens=True))# decoding generatedsummaries
    return summaries

print("Generating summaries...")
ft_summaries = generate_summaries(ft_model, ft_tokenizer, test_texts)
base_summaries = generate_summaries(base_model, base_tokenizer, test_texts)

Loading fine‑tuned and original models...
Generating summaries...


100%|██████████| 55/55 [02:04<00:00,  2.25s/it]


## Computing final metrics

In [23]:
print("Calculating final metrics:")

rouge = load("rouge") # loading all the evaluation metrics
perplexity = load("perplexity", module_type="metric")
bertscore = load("bertscore")
# computing ROUGE scores for both models
rouge_ft = rouge.compute(predictions=ft_summaries, references=references)
rouge_base = rouge.compute(predictions=base_summaries, references=references)
# filtering out empty summaries before computing perplexity
ft_filtered = [s for s in ft_summaries if s.strip()]
base_filtered = [s for s in base_summaries if s.strip()]
# computing perplexity using gpt2
perp_ft = perplexity.compute(model_id="gpt2", predictions=ft_filtered)
perp_base = perplexity.compute(model_id="gpt2", predictions=base_filtered)
# computing BERTScore
bert_ft = bertscore.compute(predictions=ft_summaries, references=references, lang="en")
bert_base = bertscore.compute(predictions=base_summaries, references=references, lang="en")

Calculating final metrics:


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Print summary table

In [24]:
print("\nEvaluation summary:") # creating formatted header
print(f"{'Model':<30}{'ROUGE-1':>9}{'ROUGE-2':>9}{'ROUGE-L':>9}{'Perplexity':>13}{'BERT F1':>10}")
print("-"*80)

def mean_f1(result): return np.mean(result["f1"]) # function to compute mean F1 from BERTScore
rows = [ # preparing data for both models
    ("Original (T5-small)", rouge_base["rouge1"], rouge_base["rouge2"], rouge_base["rougeL"], perp_base["mean_perplexity"], mean_f1(bert_base)),
    ("Fine-tuned (3 Epochs)", rouge_ft["rouge1"], rouge_ft["rouge2"], rouge_ft["rougeL"], perp_ft["mean_perplexity"], mean_f1(bert_ft))]
for r in rows: # printing each row
    print(f"{r[0]:<30}{r[1]:>9.4f}{r[2]:>9.4f}{r[3]:>9.4f}{r[4]:>13.2f}{r[5]:>10.4f}")


Evaluation summary:
Model                           ROUGE-1  ROUGE-2  ROUGE-L   Perplexity   BERT F1
--------------------------------------------------------------------------------
Original (T5-small)              0.1384   0.0130   0.1039        60.97    0.8390
Fine-tuned (3 Epochs)            0.1522   0.0184   0.1201        92.43    0.8479


## Manual comparison

In [25]:
random.seed(40) # fixed seed for reproducible results
# randomly selecting 2 essays from the test set
sample_indices = random.sample(range(len(test_texts)), 2)
for i in sample_indices:
    print("\n" + "="*50)
    print(f"Sample index: {i}")
    print("\nOriginal essay:\n", test_texts[i][:700], "...")
    print("\nGround truth summary:\n", references[i])
    print("\nOriginal model summary:\n", base_summaries[i])
    print("\nFine-Tuned model summary:\n", ft_summaries[i])
    print("\n" + "="*50)


Sample index: 234

Original essay:
 I met my first semipalmated sandpiper in a crook of Jamaica Bay, an overlooked shore strewn with broken bottles and religious offerings at the edge of New York City. I didn’t know what it was called, this small, dun-and-white bird running the flats like a wind-up toy, stopping to peck mud and racing to join another bird like itself, and then more. Soon a flock formed, several hundred fast-trotting feeders that at some secret signal took flight, wheeling with the flashing synchronisation that researchers observing starlings have mathematically likened to avalanche formation and liquids turning to gas. Entranced, I spent the afternoon watching them. The birds were too wary to approach, but if  ...

Ground truth summary:
 Animals have thoughts, feelings and personality. Why have we taken so long to catch up with animal consciousness?

Original model summary:
 semipalmated sandpipers are the most common shorebird in North America. they eat molluscs, ins